# Assignment 3a: Basic Gradio RAG Frontend
## Day 6 Session 2 - Building Simple RAG Applications

In this assignment, you'll build a simple Gradio frontend for your RAG system with just the essential features:
- Button to initialize the vector database
- Search query input and button
- Display of AI responses

**Learning Objectives:**
- Create basic Gradio interfaces
- Connect RAG backend to frontend
- Handle user interactions and database initialization
- Build functional AI-powered web applications

**Prerequisites:**
- Completed Assignment 1 (Vector Database Basics)
- Completed Assignment 2 (Advanced RAG)
- Understanding of LlamaIndex fundamentals


## 📚 Part 1: Setup and Imports

Import all necessary libraries for building your Gradio RAG application.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install -q uv
# TODO: pass your own requirements.txt file
!pip install -q "transformers>=4.40.0" "sentence-transformers>=3.0.0"
!pip install -q "torch>=2.4.0" --index-url https://download.pytorch.org/whl/cu121
!uv pip install --system -r /content/drive/MyDrive/outskill_AINE/requirements.txt   # change this to your own path of requirements.txt


Using Python 3.12.13 environment at: /usr
Resolved 198 packages in 1.11s
Prepared 14 packages in 1m 36s
Uninstalled 6 packages in 1.34s
Installed 58 packages in 551ms
 + banks==2.4.2
 + colorama==0.4.6
 + dataclasses-json==0.6.7
 + deprecated==1.3.1
 + dirtyjson==1.0.8
 + filetype==1.2.0
 + griffe==2.0.2
 + griffecli==2.0.2
 + griffelib==2.0.2
 - huggingface-hub==1.17.0
 + huggingface-hub==0.34.0
 + jedi==0.20.0
 + lance-namespace==0.8.2
 + lance-namespace-urllib3-client==0.8.2
 + lancedb==0.33.0
 + llama-cloud==1.6.0
 + llama-index==0.14.16
 + llama-index-cli==0.5.5
 + llama-index-core==0.14.22
 + llama-index-embeddings-huggingface==0.7.0
 + llama-index-embeddings-openai==0.5.2
 + llama-index-indices-managed-llama-cloud==0.11.1
 + llama-index-instrumentation==0.5.0
 + llama-index-llms-huggingface-api==0.7.0
 + llama-index-llms-openai==0.6.26
 + llama-index-llms-openai-like==0.6.0
 + llama-index-llms-openrouter==0.5.0
 + llama-index-readers-file==0.5.6
 + llama-index-readers-llama-pars

In [31]:
import os
from getpass import getpass

# securely input your key
os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter key")
print("✓ OpenRouter key set successfully")

Enter your OpenRouter key··········
✓ OpenRouter key set successfully


In [32]:
import os
import time
from pathlib import Path
from typing import List


CONFIG = {
    "llm_model": "gpt-5-mini",
    "embedding_model": "local:BAAI/bge-small-en-v1.5",
    "chunk_size": 512,
    "chunk_overlap": 50,
    "similarity_top_k": 5,
    "data_path": "/content/drive/MyDrive/outskill_AINE/data",  # change the path accordingly
    "vector_db_path": "/content/storage/assignment_multimodal_vectordb",
    "index_storage_path": "/content/storage/assignment_multimodal_index",
    "table_name": "assignment_documents"
}

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Configuration loaded")
print("Data path:", CONFIG["data_path"])
print("Vector DB path:", CONFIG["vector_db_path"])
print("Index storage path:", CONFIG["index_storage_path"])

Configuration loaded
Data path: /content/drive/MyDrive/outskill_AINE/data
Vector DB path: /content/storage/assignment_multimodal_vectordb
Index storage path: /content/storage/assignment_multimodal_index


In [33]:
# Import required libraries
import gradio as gr
import os
from pathlib import Path

# LlamaIndex components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.lancedb import LanceDBVectorStore
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openrouter import OpenRouter

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


## 🤖 Part 2: RAG Backend Class

Create a simple RAG backend that can initialize the database and answer queries.


In [34]:
class SimpleRAGBackend:
    """Simple RAG backend for Gradio frontend."""

    def __init__(self):
        self.index = None
        self.setup_settings()

    def setup_settings(self):
        """Configure LlamaIndex settings."""
        # Set up the LLM using OpenRouter
        api_key = os.getenv("OPENROUTER_API_KEY")
        if api_key:
            Settings.llm = OpenRouter(
                api_key=api_key,
                model="gpt-4o",
                temperature=0.1
            )

        # Set up the embedding model
        Settings.embed_model = OpenAIEmbedding(
            model="text-embedding-3-small",
            api_key=os.getenv("OPENROUTER_API_KEY"),
            api_base="https://openrouter.ai/api/v1"
        )

        # Set chunking parameters
        Settings.chunk_size = 512
        Settings.chunk_overlap = 50

    def initialize_database(self, data_folder="../content/drive/MyDrive/outskill_AINE/data"):
        """Initialize the vector database with documents."""
        # Check if data folder exists
        if not Path(data_folder).exists():
            return f"❌ Data folder '{data_folder}' not found!"

        try:
            # Create vector store
            vector_store = LanceDBVectorStore(
                uri="./basic_rag_vectordb",
                table_name="documents"
            )

            # Load documents
            reader = SimpleDirectoryReader(input_dir=data_folder, recursive=True)
            documents = reader.load_data()

            # Create storage context and index
            storage_context = StorageContext.from_defaults(vector_store=vector_store)
            self.index = VectorStoreIndex.from_documents(
                documents,
                storage_context=storage_context,
                show_progress=True
            )

            return f"✅ Database initialized successfully with {len(documents)} documents!"

        except Exception as e:
            return f"❌ Error initializing database: {str(e)}"

    def query(self, question):
        """Query the RAG system and return response."""
        # Check if index exists
        if self.index is None:
            return "❌ Please initialize the database first!"

        # Check if question is empty
        if not question or not question.strip():
            return "⚠️ Please enter a question first!"

        try:
            # Create query engine and get response
            query_engine = self.index.as_query_engine()
            response = query_engine.query(question)
            return str(response)

        except Exception as e:
            return f"❌ Error processing query: {str(e)}"

# Initialize the backend
rag_backend = SimpleRAGBackend()
print("🚀 RAG Backend initialized and ready!")


🚀 RAG Backend initialized and ready!


In [28]:
class SimpleRAGBackend:
    """Simple RAG backend for Gradio frontend."""

    def __init__(self):
        self.index = None
        self.setup_settings()

    def setup_settings(self):
        """Configure LlamaIndex settings."""
        # Set up the LLM using OpenRouter
        openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
        if openrouter_api_key:
            Settings.llm = OpenRouter(
                api_key=openrouter_api_key,
                model="gpt-4o",
                temperature=0.1
            )

        # Set up the embedding model with OpenAI API key
        openai_api_key = os.getenv("OPENAI_API_KEY")
        if openai_api_key:
            Settings.embed_model = OpenAIEmbedding(
                model="text-embedding-3-small",
                api_key=openai_api_key
            )
        else:
            print("⚠️ OpenAI API key not found. Embedding model not configured.")

        # Set chunking parameters
        Settings.chunk_size = 512
        Settings.chunk_overlap = 50

    def initialize_database(self, data_folder="../content/drive/MyDrive/outskill_AINE/data"):
        """Initialize the vector database with documents."""
        # Check if data folder exists
        if not Path(data_folder).exists():
            return f"❌ Data folder '{data_folder}' not found!"

        # Ensure embedding model is configured before proceeding
        if Settings.embed_model is None:
            return "❌ Embedding model not configured. Please provide an OpenAI API key."

        try:
            # Create vector store
            vector_store = LanceDBVectorStore(
                uri="./basic_rag_vectordb",
                table_name="documents"
            )

            # Load documents
            reader = SimpleDirectoryReader(input_dir=data_folder, recursive=True)
            documents = reader.load_data()

            # Create storage context and index
            storage_context = StorageContext.from_defaults(vector_store=vector_store)
            self.index = VectorStoreIndex.from_documents(
                documents,
                storage_context=storage_context,
                show_progress=True
            )

            return f"✅ Database initialized successfully with {len(documents)} documents!"

        except Exception as e:
            return f"❌ Error initializing database: {str(e)}"

    def query(self, question):
        """Query the RAG system and return response."""
        # Check if index exists
        if self.index is None:
            return "❌ Please initialize the database first!"

        # Check if question is empty
        if not question or not question.strip():
            return "⚠️ Please enter a question first!"

        try:
            # Create query engine and get response
            query_engine = self.index.as_query_engine()
            response = query_engine.query(question)
            return str(response)

        except Exception as e:
            return f"❌ Error processing query: {str(e)}"

# Initialize the backend
rag_backend = SimpleRAGBackend()
print("🚀 RAG Backend initialized and ready!")

🚀 RAG Backend initialized and ready!


## 🎨 Part 3: Gradio Interface

Create a simple Gradio interface with:
1. Button to initialize the database
2. Text input for queries
3. Button to submit queries
4. Text output for responses
5. Text output for status messages


In [35]:
def create_basic_rag_interface():
    """Create basic RAG interface with essential features."""

    def initialize_db():
        """Handle database initialization."""
        return rag_backend.initialize_database()

    def handle_query(question):
        """Handle user queries."""
        return rag_backend.query(question)

    # TODO: Create Gradio interface using gr.Blocks()
    # Hint: Look at the structure below and fill in the missing components

    with gr.Blocks(title="Basic RAG Assistant") as interface:
        # TODO: Add title and description
        # Hint: Use gr.Markdown() for formatted text
        gr.Markdown("# Basic. RAG Assistant")
        gr.Markdown("Ask questions about your documents using AI-powered retrieval.")

        # TODO: Add initialization section
        # Hint: You need to use gr.Button to initialize the database
        init_btn = gr.Button("Initialize Database", variant="primary")

        # TODO: Add status output
        # Hint: You need to use gr.Textbox to display the status

        # The connection between the button and the status output has already been implemented
        # at the end of this function

        status_output = gr.Textbox(label="Status", interactive=False)

        # TODO: Add query section
        # Hint: You need a text input, submit button, and response output

        # Use gr.Textbox to create a text input
        query_input = gr.Textbox(label="Your Question", placeholder="Ask anything which you want to know from document...")

        # Use gr.Button to create a submit button
        submit_btn = gr.Button("Ask Question", variant="secondary")

        # Use gr.Textbox to create a response output
        response_output = gr.Textbox(label="AI Response", lines=10)

        # Connect buttons to functions
        # Uncomment when above is implemented
        init_btn.click(initialize_db, outputs=[status_output])
        submit_btn.click(handle_query, inputs=[query_input], outputs=[response_output])

    return interface

# Create the interface
basic_interface = create_basic_rag_interface()
print("✅ Basic RAG interface created successfully!")


✅ Basic RAG interface created successfully!


## 🚀 Part 4: Launch Your Application

Launch your Gradio application and test it!


In [36]:
print("🎉 Launching your Basic RAG Assistant...")
print("🔗 Your application will open in a new browser tab!")
print("")
print("📋 Testing Instructions:")
print("1. Click 'Initialize Database' button first")
print("2. Wait for success message")
print("3. Enter a question in the query box")
print("4. Click 'Ask Question' to get AI response")
print("")
print("💡 Example questions to try:")
print("- What are the main topics in the documents?")
print("- Summarize the key findings")
print("- Explain the methodology used")
print("")
print("🚀 Launch your app:")

# Your launch code here:
# Uncomment when implemented
basic_interface.launch()

🎉 Launching your Basic RAG Assistant...
🔗 Your application will open in a new browser tab!

📋 Testing Instructions:
1. Click 'Initialize Database' button first
2. Wait for success message
3. Enter a question in the query box
4. Click 'Ask Question' to get AI response

💡 Example questions to try:
- What are the main topics in the documents?
- Summarize the key findings
- Explain the methodology used

🚀 Launch your app:
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://78011f71de5afe0f24.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## ✅ Assignment Completion Checklist

Before submitting, ensure you have:

- [x] RAG backend is provided and working
- [ ] Created Gradio interface with required components:
  - [ ] Title and description using gr.Markdown()
  - [ ] Initialize database button using gr.Button()
  - [ ] Status output using gr.Textbox()
  - [ ] Query input field using gr.Textbox()
  - [ ] Submit query button using gr.Button()
  - [ ] Response output area using gr.Textbox()
- [ ] Connected buttons to backend functions using .click()
- [ ] Successfully launched the application
- [ ] Tested the full workflow (initialize → query → response)

## 🎊 Congratulations!

You've successfully built your first Gradio RAG application! You now have:

- A functional web interface for your RAG system
- Understanding of Gradio basics and component connections
- A foundation for building more complex AI applications

**Next Steps**: Complete Assignment 3b to add advanced configuration options to your RAG interface!
